In [1]:
!pip install ortools

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 29.8/29.8 MB 10.4 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [ortools]m3/4 [ortools]


In [2]:
import pandas as pd
import numpy as np

from ortools.sat.python import cp_model

print("OR-Tools imported successfully.")

OR-Tools imported successfully.


In [3]:
decision_data = pd.read_csv(
    "../data/predictions/multi_objective_decision_scores.csv"
)

print("Decision dataset shape:", decision_data.shape)

display(
    decision_data.head(10)
)

Decision dataset shape: (6, 32)


,decision_rank,priority_rank,task_id,asset_id,section_id,department,maintenance_type,defect_type,severity,criticality,...,affected_trains_estimate,failure_risk_factor,urgency_factor,criticality_factor,overdue_factor,duration_factor,traffic_factor,operational_factor,maintenance_decision_score,decision_category
0,1,6,SMMS002,SIG002,AGC-GWL-01,S&T,INSPECTION,SIGNAL_CHECK,3,7,...,5.074511,0.0,0.365000,0.7,0.000000,0.166667,0.842983,0.611711,0.362388,MEDIUM
1,2,5,TDMS002,OHE002,JHS-BINA-01,TRACTION,INSPECTION,OHE_CHECK,3,8,...,5.736522,0.0,0.400000,0.8,0.000000,0.250000,0.635305,0.572357,0.361884,MEDIUM
2,3,1,TMS001,TRK001,NDL-MTJ-01,ENGINEERING,REPAIR,RAIL_CRACK,9,10,...,0.000000,0.0,0.718333,1.0,0.033333,0.500000,0.000000,0.000000,0.322000,MEDIUM
3,4,2,SMMS001,SIG001,MTJ-AGC-01,S&T,REPAIR,SIGNAL_DEGRADATION,8,9,...,0.000000,0.0,0.635000,0.9,0.000000,0.250000,0.000000,0.000000,0.274500,MEDIUM
4,5,3,TDMS001,OHE001,GWL-JHS-01,TRACTION,REPAIR,OHE_INSULATOR,7,9,...,0.000000,0.0,0.595000,0.9,0.000000,0.333333,0.000000,0.000000,0.270667,MEDIUM
5,6,4,TMS002,TRK002,NDL-MTJ-02,ENGINEERING,INSPECTION,TRACK_WEAR,4,8,...,0.000000,0.0,0.440000,0.8,0.000000,0.333333,0.000000,0.000000,0.224667,LOW


In [4]:
print(
    decision_data.columns.tolist()
)

['decision_rank', 'priority_rank', 'task_id', 'asset_id', 'section_id', 'department', 'maintenance_type', 'defect_type', 'severity', 'criticality', 'overdue_days', 'estimated_duration', 'required_manpower', 'status', 'failure_probability', 'urgency_score', 'priority_score', 'priority_category', 'predicted_delay_minutes', 'operational_impact_score', 'operational_impact_category', 'traffic_intensity', 'affected_trains_estimate', 'failure_risk_factor', 'urgency_factor', 'criticality_factor', 'overdue_factor', 'duration_factor', 'traffic_factor', 'operational_factor', 'maintenance_decision_score', 'decision_category']


In [5]:
numeric_columns = [
    "estimated_duration",
    "required_manpower",
    "maintenance_decision_score",
    "criticality",
    "overdue_days",
    "predicted_delay_minutes"
]

for column in numeric_columns:
    if column in decision_data.columns:
        decision_data[column] = pd.to_numeric(
            decision_data[column],
            errors="coerce"
        )

decision_data["estimated_duration"] = (
    decision_data["estimated_duration"]
    .fillna(60)
    .clip(lower=15)
)

decision_data["required_manpower"] = (
    decision_data["required_manpower"]
    .fillna(1)
    .clip(lower=1)
)

decision_data["maintenance_decision_score"] = (
    decision_data["maintenance_decision_score"]
    .fillna(0)
    .clip(0, 1)
)

decision_data["criticality"] = (
    decision_data["criticality"]
    .fillna(5)
    .clip(0, 10)
)

decision_data["overdue_days"] = (
    decision_data["overdue_days"]
    .fillna(0)
    .clip(lower=0)
)

decision_data["predicted_delay_minutes"] = (
    decision_data["predicted_delay_minutes"]
    .fillna(0)
    .clip(lower=0)
)

print("Optimizer inputs cleaned.")

Optimizer inputs cleaned.


In [6]:
optimizer_tasks = (
    decision_data
    .sort_values(
        "maintenance_decision_score",
        ascending=False
    )
    .head(30)
    .reset_index(drop=True)
)

print(
    "Candidate tasks:",
    len(optimizer_tasks)
)

display(
    optimizer_tasks[
        [
            "task_id",
            "section_id",
            "department",
            "estimated_duration",
            "required_manpower",
            "maintenance_decision_score"
        ]
    ]
)

Candidate tasks: 6


,task_id,section_id,department,estimated_duration,required_manpower,maintenance_decision_score
0,SMMS002,AGC-GWL-01,S&T,30,2,0.362388
1,TDMS002,JHS-BINA-01,TRACTION,45,4,0.361884
2,TMS001,NDL-MTJ-01,ENGINEERING,90,8,0.322000
3,SMMS001,MTJ-AGC-01,S&T,45,3,0.274500
4,TDMS001,GWL-JHS-01,TRACTION,60,5,0.270667
5,TMS002,NDL-MTJ-02,ENGINEERING,60,5,0.224667


In [7]:
SLOT_MINUTES = 30

PLANNING_HOURS = 6

NUM_SLOTS = (
    PLANNING_HOURS * 60
) // SLOT_MINUTES

print("Slot length:", SLOT_MINUTES, "minutes")
print("Number of slots:", NUM_SLOTS)

Slot length: 30 minutes
Number of slots: 12


In [8]:
block_windows = [
    {
        "block_id": "B001",
        "start_slot": 1,
        "end_slot": 5
    },
    {
        "block_id": "B002",
        "start_slot": 6,
        "end_slot": 10
    }
]

print("Available block windows:")

for block in block_windows:
    print(
        block["block_id"],
        "slots",
        block["start_slot"],
        "to",
        block["end_slot"]
    )

Available block windows:
B001 slots 1 to 5
B002 slots 6 to 10


In [9]:
model = cp_model.CpModel()

print("CP-SAT model created.")

CP-SAT model created.


In [10]:
task_selected = {}

for i, row in optimizer_tasks.iterrows():
    task_selected[i] = model.NewBoolVar(
        f"selected_{i}"
    )

print(
    "Created selection variables for",
    len(task_selected),
    "tasks."
)

Created selection variables for 6 tasks.


In [11]:
task_start = {}

for i, row in optimizer_tasks.iterrows():
    task_start[i] = model.NewIntVar(
        0,
        NUM_SLOTS - 1,
        f"start_{i}"
    )

print("Start-time variables created.")

Start-time variables created.


In [12]:
optimizer_tasks["duration_slots"] = np.ceil(
    optimizer_tasks["estimated_duration"]
    / SLOT_MINUTES
).astype(int)

optimizer_tasks["duration_slots"] = (
    optimizer_tasks["duration_slots"]
    .clip(lower=1)
)

display(
    optimizer_tasks[
        [
            "task_id",
            "estimated_duration",
            "duration_slots"
        ]
    ]
)

,task_id,estimated_duration,duration_slots
0,SMMS002,30,1
1,TDMS002,45,2
2,TMS001,90,3
3,SMMS001,45,2
4,TDMS001,60,2
5,TMS002,60,2


In [13]:
task_end = {}

for i, row in optimizer_tasks.iterrows():

    duration = int(
        row["duration_slots"]
    )

    task_end[i] = model.NewIntVar(
        0,
        NUM_SLOTS,
        f"end_{i}"
    )

    model.Add(
        task_end[i]
        == task_start[i] + duration
    )

print("Task end-time variables created.")

Task end-time variables created.


In [14]:
for i, row in optimizer_tasks.iterrows():

    model.Add(
        task_end[i] <= NUM_SLOTS
    ).OnlyEnforceIf(
        task_selected[i]
    )

print(
    "Planning-window constraints added."
)

Planning-window constraints added.


In [15]:
for i, row in optimizer_tasks.iterrows():

    model.Add(
        task_end[i] <= NUM_SLOTS
    ).OnlyEnforceIf(
        task_selected[i]
    )

print(
    "Planning-window constraints added."
)

Planning-window constraints added.


In [17]:
# Recreate block availability
block_windows = [
    {
        "block_id": "B001",
        "start_slot": 1,
        "end_slot": 5
    },
    {
        "block_id": "B002",
        "start_slot": 6,
        "end_slot": 10
    }
]

# Create set of all available time slots
available_slots = set()

for block in block_windows:
    for slot in range(
        block["start_slot"],
        block["end_slot"]
    ):
        available_slots.add(slot)

print("Available slots:")
print(sorted(available_slots))


# Add block-availability constraints
for i, row in optimizer_tasks.iterrows():

    duration = int(
        row["duration_slots"]
    )

    allowed_starts = []

    for start in range(NUM_SLOTS):

        end = start + duration

        if end > NUM_SLOTS:
            continue

        # Every slot occupied by this task
        # must be inside an available block
        if all(
            slot in available_slots
            for slot in range(start, end)
        ):
            allowed_starts.append(start)

    if allowed_starts:

        model.AddAllowedAssignments(
            [task_start[i]],
            [
                [start]
                for start in allowed_starts
            ]
        ).OnlyEnforceIf(
            task_selected[i]
        )

    else:

        model.Add(
            task_selected[i] == 0
        )

print("Block availability constraints added successfully.")

Available slots:
[1, 2, 3, 4, 6, 7, 8, 9]
Block availability constraints added successfully.


In [18]:
task_intervals = {}

for i, row in optimizer_tasks.iterrows():

    duration = int(
        row["duration_slots"]
    )

    task_intervals[i] = model.NewOptionalIntervalVar(
        task_start[i],
        duration,
        task_end[i],
        task_selected[i],
        f"interval_{i}"
    )

print(
    "Created",
    len(task_intervals),
    "optional task intervals."
)

Created 6 optional task intervals.


In [19]:
task_intervals = {}

for i, row in optimizer_tasks.iterrows():

    duration = int(row["duration_slots"])

    task_intervals[i] = model.NewOptionalIntervalVar(
        task_start[i],
        duration,
        task_end[i],
        task_selected[i],
        f"interval_{i}"
    )

print(
    "Created",
    len(task_intervals),
    "optional task intervals."
)

Created 6 optional task intervals.


In [20]:
model.AddNoOverlap(
    list(task_intervals.values())
)

print("No-overlap constraint added successfully.")

No-overlap constraint added successfully.


In [21]:
MAX_MANPOWER = 12

manpower_intervals = []
manpower_demands = []

for i, row in optimizer_tasks.iterrows():

    duration = int(row["duration_slots"])
    manpower = int(row["required_manpower"])

    interval = model.NewOptionalIntervalVar(
        task_start[i],
        duration,
        task_end[i],
        task_selected[i],
        f"manpower_interval_{i}"
    )

    manpower_intervals.append(interval)
    manpower_demands.append(manpower)

model.AddCumulative(
    manpower_intervals,
    manpower_demands,
    MAX_MANPOWER
)

print("Manpower capacity constraint added successfully.")

Manpower capacity constraint added successfully.


In [22]:
for section_id, group in optimizer_tasks.groupby("section_id"):

    task_ids = group.index.tolist()

    section_intervals = [
        task_intervals[i]
        for i in task_ids
    ]

    model.AddNoOverlap(
        section_intervals
    )

print("Section conflict constraints added successfully.")

Section conflict constraints added successfully.


In [23]:
optimizer_tasks["coordination_group"] = (
    optimizer_tasks["section_id"].astype(str)
    + "_"
    + optimizer_tasks["department"].astype(str)
)

print("Coordination groups created.")

display(
    optimizer_tasks[
        [
            "task_id",
            "section_id",
            "department",
            "coordination_group"
        ]
    ].head(20)
)

Coordination groups created.


,task_id,section_id,department,coordination_group
0,SMMS002,AGC-GWL-01,S&T,AGC-GWL-01_S&T
1,TDMS002,JHS-BINA-01,TRACTION,JHS-BINA-01_TRACTION
2,TMS001,NDL-MTJ-01,ENGINEERING,NDL-MTJ-01_ENGINEERING
3,SMMS001,MTJ-AGC-01,S&T,MTJ-AGC-01_S&T
4,TDMS001,GWL-JHS-01,TRACTION,GWL-JHS-01_TRACTION
5,TMS002,NDL-MTJ-02,ENGINEERING,NDL-MTJ-02_ENGINEERING


In [24]:
department_counts = (
    optimizer_tasks["department"]
    .value_counts()
)

print("Candidate tasks by department:")
print(department_counts)

Candidate tasks by department:
department
S&T            2
TRACTION       2
ENGINEERING    2
Name: count, dtype: int64


In [25]:
SCORE_SCALE = 1000

priority_coefficients = {}

for i, row in optimizer_tasks.iterrows():

    priority = int(
        round(
            row["maintenance_decision_score"]
            * SCORE_SCALE
        )
    )

    priority_coefficients[i] = priority

print("Priority coefficients created.")

print(
    list(priority_coefficients.items())[:10]
)

Priority coefficients created.
[(0, 362), (1, 362), (2, 322), (3, 274), (4, 271), (5, 225)]


In [26]:
SCORE_SCALE = 1000

priority_coefficients = {}

for i, row in optimizer_tasks.iterrows():

    priority = int(
        round(
            row["maintenance_decision_score"]
            * SCORE_SCALE
        )
    )

    priority_coefficients[i] = priority

print("Priority coefficients created.")

print(
    list(priority_coefficients.items())[:10]
)

Priority coefficients created.
[(0, 362), (1, 362), (2, 322), (3, 274), (4, 271), (5, 225)]


In [27]:
IMPACT_SCALE = 10

objective_terms = []

for i, row in optimizer_tasks.iterrows():

    priority = priority_coefficients[i]

    delay_penalty = int(
        round(
            row["predicted_delay_minutes"]
            * IMPACT_SCALE
        )
    )

    coefficient = (
        priority
        - delay_penalty
    )

    objective_terms.append(
        coefficient * task_selected[i]
    )

model.Maximize(
    sum(objective_terms)
)

print("Optimization objective created successfully.")

Optimization objective created successfully.


In [28]:
from ortools.sat.python import cp_model

solver = cp_model.CpSolver()

solver.parameters.max_time_in_seconds = 30
solver.parameters.num_search_workers = 8

print("CP-SAT solver configured successfully.")

CP-SAT solver configured successfully.


In [29]:
status = solver.Solve(model)

print("Solver status:", solver.StatusName(status))

Solver status: OPTIMAL


In [30]:
selected_rows = []

for i, row in optimizer_tasks.iterrows():

    if solver.Value(task_selected[i]) == 1:

        start_slot = solver.Value(task_start[i])
        end_slot = solver.Value(task_end[i])

        selected_rows.append({
            "task_id": row["task_id"],
            "section_id": row["section_id"],
            "department": row["department"],
            "start_slot": start_slot,
            "end_slot": end_slot,
            "duration_minutes": (
                end_slot - start_slot
            ) * SLOT_MINUTES,
            "required_manpower": row["required_manpower"],
            "maintenance_decision_score": (
                row["maintenance_decision_score"]
            ),
            "predicted_delay_minutes": (
                row["predicted_delay_minutes"]
            )
        })

optimized_plan = pd.DataFrame(selected_rows)

optimized_plan = optimized_plan.sort_values(
    ["start_slot", "section_id"]
).reset_index(drop=True)

display(optimized_plan)

,task_id,section_id,department,start_slot,end_slot,duration_minutes,required_manpower,maintenance_decision_score,predicted_delay_minutes
0,TMS001,NDL-MTJ-01,ENGINEERING,2,5,90,8,0.322000,0.0
1,SMMS001,MTJ-AGC-01,S&T,6,8,60,3,0.274500,0.0
2,TDMS001,GWL-JHS-01,TRACTION,8,10,60,5,0.270667,0.0


In [31]:
optimized_plan.to_csv(
    "../data/predictions/optimized_block_plan.csv",
    index=False
)

print(
    "Optimized block plan saved successfully:"
)

print(
    "../data/predictions/optimized_block_plan.csv"
)

Optimized block plan saved successfully:
../data/predictions/optimized_block_plan.csv
